### Formulation of the problem (Step 3 — Center Relocation)

We now let the **locations of the SR offices (“center bricks”) be decision variables**.

#### Sets and data

- $B = \{1,\dots,m\}$ : set of bricks (here $m = 22$)  
- $S = B$ : any brick can host an SR office  
- $w_i$ : workload associated with brick $i$  
- $d_{i,j}$ : distance between brick $i$ and a center located at brick $j$  

The **current offices** are located at bricks:

- SR1: brick 4  
- SR2: brick 14  
- SR3: brick 16  
- SR4: brick 22  

We denote this initial set of offices by $C^0 = \{4, 14, 16, 22\}$.

#### Decision variables

- $x_{i,j} \in \{0,1\}$ : brick $i$ is served by an office located at brick $j$  
- $y_j \in \{0,1\}$ : an office is opened at brick $j$  
- $wm \ge 0$ : maximum workload over all SRs (workload fairness indicator)

#### Constraints

1. **Each brick is assigned to exactly one office**

$$
\sum_{j \in S} x_{i,j} = 1 \quad \forall i \in B
$$

2. **Exactly $n$ offices are opened (here $n = 4$)**

$$
\sum_{j \in S} y_j = n
$$

3. **No assignment to a closed office**

$$
x_{i,j} \le y_j \quad \forall i \in B,\ \forall j \in S
$$

4. **Definition of the maximum workload**

If we denote by $L_j = \sum_i w_i x_{i,j}$ the workload of the office at brick $j$, we enforce:

$$
wm \ge L_j \quad \forall j \in S
$$

so that $wm$ is indeed the **maximal workload** over open offices.

---

#### Objectives

1. **Total distance (to be minimized)**

$$
\text{Dist}(x) = \sum_{i \in B} \sum_{j \in S} d_{i,j} \, x_{i,j}
$$

2. **Workload fairness (to be minimized as $wm$)**

$$
\text{Fairness}(x) = wm = \max_j L_j
$$

We first study the **bi-objective problem** (Dist, $wm$) using an **ε-constraint** approach on $wm$.

---

#### Disruption as “number of relocated offices”

In Step 3, disruption is redefined as the **number of offices that change location** compared to the initial set $C^0$:

- An office is *relocated* if its initial brick is no longer an office in the new solution.  
- The number of relocated offices therefore takes integer values in $[0, n]$.

This gives a **three-objective problem**:

1. Total distance (min)  
2. Workload fairness $wm$ (min)  
3. Number of relocated offices (min)

We will generate and visualize **non-dominated solutions** for this 3‑objective problem.


In [13]:
from gurobipy import Model, GRB, quicksum
import gurobipy as gp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# Data for the 22-brick instance (center relocation)
# ---------------------------

# Distance matrix between bricks: d[i,j] = distance if the office is at brick j
# We reuse the same distances as in Step 1, read from the Excel file.
distances = pd.read_excel("distances.xlsx", sheet_name=1, header=None).iloc[2:, 2:]
distances = distances.reset_index(drop=True).reset_index(drop=True)

m = len(distances)   # number of bricks (22)
n = 4                # number of SRs / offices to open

# Workloads (same as in Step 1), scaled so that sum(workloads) = 4
workloads = [
    0.1609, 0.1164, 0.1026, 0.1516, 0.0939,
    0.1320, 0.0687, 0.0930, 0.2116, 0.2529,
    0.0868, 0.0828, 0.0975, 0.8177, 0.4115,
    0.3795, 0.0710, 0.0427, 0.1043, 0.0997,
    0.1698, 0.2531,
]
assert abs(sum(workloads) - 4.0) < 1e-6, "Workloads should sum to 4"

# Initial center locations from Step 1 (1-based indices)
initial_centers = [4, 14, 16, 22]
initial_centers_zero_based = [c - 1 for c in initial_centers]


## Bi-objective exploration: total distance vs maximal workload


In [14]:
def build_center_model(dist_df, workloads, n_centers):
    """Build the base Gurobi model for Step 3 (center relocation).

    Variables
    ---------
    x[i,j] : 1 if brick i is served by an office at brick j
    y[j]   : 1 if an office is opened at brick j
    wm     : max workload over all offices

    Constraints
    -----------
    - each brick is assigned to exactly one office
    - exactly n_centers offices are opened
    - no assignment to a closed office
    - wm >= workload_j  for all j

    The model is returned **without** an objective so that we can
    reuse it for different mono- and bi-objective runs.
    """
    m = len(workloads)
    model = Model("step3_centers")
    model.Params.OutputFlag = 0

    # Decision variables
    x = model.addVars(m, m, vtype=GRB.BINARY, name="x")
    y = model.addVars(m, vtype=GRB.BINARY, name="y")
    wm = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="wm")

    # Each brick is assigned to exactly one office
    for i in range(m):
        model.addConstr(quicksum(x[i, j] for j in range(m)) == 1,
                        name=f"assign_brick_{i}")

    # Exactly n_centers offices are opened
    model.addConstr(quicksum(y[j] for j in range(m)) == n_centers,
                    name="nb_centers")

    # No assignment to a closed office
    for i in range(m):
        for j in range(m):
            model.addConstr(x[i, j] <= y[j], name=f"open_link_{i}_{j}")

    # wm >= workload of each office
    for j in range(m):
        model.addConstr(
            wm >= quicksum(workloads[i] * x[i, j] for i in range(m)),
            name=f"wm_ge_load_{j}"
        )

    # Store a handy expression for the total distance
    dist_expr = quicksum(
        float(dist_df.iloc[i, j]) * x[i, j]
        for i in range(m) for j in range(m)
    )
    model._dist_expr = dist_expr  # attach as custom attribute
    model._x = x
    model._y = y
    model._wm = wm

    return model


def extract_solution(model):
    """Return a compact summary of a solved model solution.

    Returns a dict with:
      - centers: list of center bricks (1-based indices)
      - loads: workload at each open center
      - total_distance
      - wm_value (computed max of loads)
    """
    m = len(workloads)
    x = model._x
    y = model._y
    wm = model._wm          # still available if you need it later
    dist_expr = model._dist_expr

    centers = [j for j in range(m) if y[j].X > 0.5]
    centers_1_based = [j + 1 for j in centers]

    loads = []
    for j in centers:
        load_j = sum(workloads[i] * x[i, j].X for i in range(m))
        loads.append((j + 1, load_j))

    total_distance = dist_expr.getValue()
    # true max workload across open centers
    wm_value = max(load_j for _, load_j in loads) if loads else 0.0

    return {
        "centers": centers_1_based,
        "loads": loads,
        "total_distance": total_distance,
        "wm": wm_value,
    }



def print_solution(tag, sol):
    print(f"--- {tag} ---")
    print(f"Centers (bricks) : {sol['centers']}")
    for j, load_j in sol["loads"]:
        print(f"  Center at brick {j:2d} -> workload = {load_j:.5f}")
    print(f"Total distance   = {sol['total_distance']:.5f}")
    print(f"Max workload wm  = {sol['wm']:.5f}\n")


# 1) Minimize total distance only (base model for ε-constraint)
base_model = build_center_model(distances, workloads, n_centers=n)
base_model.setObjective(base_model._dist_expr, GRB.MINIMIZE)
base_model.optimize()
sol_dist = extract_solution(base_model)
print_solution("Distance minimization", sol_dist)


# 2) Minimize maximal workload only
model_wm = build_center_model(distances, workloads, n_centers=n)
model_wm.setObjective(model_wm._wm, GRB.MINIMIZE)
model_wm.optimize()
sol_wm = extract_solution(model_wm)
print_solution("Max workload minimization", sol_wm)


--- Distance minimization ---
Centers (bricks) : [2, 6, 12, 15]
  Center at brick  2 -> workload = 1.00680
  Center at brick  6 -> workload = 0.75080
  Center at brick 12 -> workload = 0.08280
  Center at brick 15 -> workload = 2.15960
Total distance   = 88.30170
Max workload wm  = 2.15960

--- Max workload minimization ---
Centers (bricks) : [1, 2, 3, 4]
  Center at brick  1 -> workload = 1.00010
  Center at brick  2 -> workload = 1.00010
  Center at brick  3 -> workload = 1.00000
  Center at brick  4 -> workload = 0.99980
Total distance   = 407.28277
Max workload wm  = 1.00010



In [ ]:
# ε-constraint: generate the distance vs wm Pareto curve

# First, compute the best possible wm (ideal point on fairness)
wm_model = build_center_model(distances, workloads, n_centers=n)
wm_model.setObjective(wm_model._wm, GRB.MINIMIZE)
wm_model.optimize()
wm_min = wm_model._wm.X

# A trivial upper bound is the sum of all workloads (all bricks to one SR)
wm_max = sum(workloads)

print(f"wm_min = {wm_min:.3f}, wm_max = {wm_max:.3f}")

# Grid of allowed wm values (ε) between wm_min and wm_max
epsilon_step = 0.01      # Step 0.01
eps_grid = np.arange(wm_min, wm_max + 1e-9, epsilon_step)

pareto_points = []   # list of (wm_val, dist_val)

for i, eps in enumerate(eps_grid):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(eps_grid)}] Testing wm <= {eps:.3f}")
    print(f"{'='*60}")
    
    # Copy the base distance model (same structure, warm start)
    m2 = base_model.copy()
    m2.Params.OutputFlag = 1  # Enable Gurobi output
    m2.Params.TimeLimit = 60  # 60 seconds max per solve

    # Get the wm variable in this copy
    wm_var = m2.getVarByName("wm")

    # Cap wm by ε
    m2.addConstr(wm_var <= eps + 1e-9, name=f"wm_cap_{eps:.3f}")

    # Rebuild the distance expression using variables of m2
    dist2 = quicksum(
        float(distances.iloc[i_brick, j_brick]) * m2.getVarByName(f"x[{i_brick},{j_brick}]")
        for i_brick in range(m) for j_brick in range(m)
    )

    # Objective : minimize distance (with tiny tie-break on wm)
    m2.setObjective(dist2 + 1e-9 * wm_var, GRB.MINIMIZE)
    m2.optimize()

    if m2.Status == GRB.OPTIMAL:
        wm_val = wm_var.X
        dist_val = dist2.getValue()
        pareto_points.append((wm_val, dist_val))
        print(f"\nSOLUTION FOUND: wm = {wm_val:.3f}, distance = {dist_val:.2f}")
    elif m2.Status == GRB.TIME_LIMIT:
        print(f"\nTIME LIMIT REACHED – partial solution may exist")
    else:
        print(f"\nNO SOLUTION (Status code: {m2.Status})")


print(f"\n{'='*60}")
print(f"SUMMARY: Collected {len(pareto_points)} feasible solutions")
print(f"{'='*60}\n")

# Filter non-dominated points in (wm, distance)
pareto_points = sorted(set(pareto_points))  # remove duplicates
pareto_nd = []
for wm_val, dist_val in pareto_points:
    if not any(
        (wm2 <= wm_val + 1e-9 and dist2 <= dist_val + 1e-9 and (wm2 < wm_val - 1e-9 or dist2 < dist_val - 1e-9))
        for wm2, dist2 in pareto_points
        if (wm2, dist2) != (wm_val, dist_val)
    ):
        pareto_nd.append((wm_val, dist_val))

pareto_nd.sort()

print(f"Non-dominated points: {len(pareto_nd)}\n")

# Plot the 2D Pareto frontier
if pareto_nd:
    xs = [p[0] for p in pareto_nd]  # wm
    ys = [p[1] for p in pareto_nd]  # distance
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys, "o-", linewidth=1)
    plt.xlabel("Max workload per SR (wm)")
    plt.ylabel("Total distance")
    plt.title("Step 3 – Distance vs wm (ε-constraint Pareto frontier)")
    plt.grid(True)
    plt.show()
else:
    print("No feasible Pareto points found.")

wm_min = 1.000, wm_max = 4.000

[1/300] Testing wm <= 1.000
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900HX, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  60

Optimize a model with 530 rows, 507 columns and 1981 nonzeros
Model fingerprint: 0x0f0f40d0
Variable types: 1 continuous, 506 integer (506 binary)
Coefficient statistics:
  Matrix range     [4e-02, 1e+00]
  Objective range  [1e-09, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Presolve removed 23 rows and 0 columns
Presolve time: 0.01s
Presolved: 507 rows, 507 columns, 2046 nonzeros
Variable types: 1 continuous, 506 integer (506 binary)

Root relaxation: objective 9.733293e+01, 197 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Curren

## Three-objective analysis: distance, workload fairness, and relocated offices


In [ ]:
def compute_relocated_expression(model, initial_centers_zero_based):
    """Return a linear expression for the number of relocated offices.

    relocated = n - sum_{j in C0} y_j
    """
    kept_expr = quicksum(model.getVarByName(f"y[{j}]") for j in initial_centers_zero_based)
    relocated_expr = len(initial_centers_zero_based) - kept_expr
    return relocated_expr


print("\n" + "="*70)
print("THREE-OBJECTIVE ANALYSIS: Distance vs Workload vs Relocations")
print("="*70 + "\n")

# Collect all (distance, wm, reloc) triples from ε-constraint runs
three_obj_points = []  # list of (dist, wm, reloc)

# We explore all possible relocation budgets k = 0..n
for k in range(n + 1):
    print(f"\n{'#'*70}")
    print(f"# RELOCATION BUDGET: k = {k} (max {k} offices can be relocated)")
    print(f"{'#'*70}\n")
    
    # 1) Find wm_min under this relocation budget (ideal fairness)
    print(f"[Step 1/2] Finding minimum wm with relocation budget k={k}...")
    m_wm = base_model.copy()
    m_wm.Params.OutputFlag = 0  # Suppress solver log
    m_wm.Params.TimeLimit = 30
    wm_var_wm = m_wm.getVarByName("wm")
    reloc_expr = compute_relocated_expression(m_wm, initial_centers_zero_based)
    m_wm.addConstr(reloc_expr <= k + 1e-6, name=f"reloc_le_{k}")
    m_wm.setObjective(wm_var_wm, GRB.MINIMIZE)
    m_wm.optimize()
    
    if m_wm.Status != GRB.OPTIMAL:
        print(f"  Could not find minimum wm for k={k} (Status: {m_wm.Status})")
        print(f"  Skipping this relocation budget.\n")
        continue
    
    wm_min_k = wm_var_wm.X
    print(f"  Minimum wm for k={k}: {wm_min_k:.3f}\n")


    # 2) ε-grid for this k
    # Grid of allowed wm values (ε) between wm_min_k and wm_max
    epsilon_step = 0.01      # Step 0.01
    eps_grid_k = np.arange(wm_min_k, wm_max + 1e-9, epsilon_step)

    print(f"[Step 2/2] Exploring {len(eps_grid_k)} wm values between {wm_min_k:.3f} and {wm_max:.3f}...\n")


    for i_eps, eps in enumerate(eps_grid_k):
        print(f"  [{i_eps+1}/{len(eps_grid_k)}] k={k}, wm<={eps:.3f}... ", end="", flush=True)
        
        m2 = base_model.copy()
        m2.Params.OutputFlag = 0  # Suppress solver log
        m2.Params.TimeLimit = 30  # 30 seconds per solve
        
        wm_var = m2.getVarByName("wm")
        reloc_expr2 = compute_relocated_expression(m2, initial_centers_zero_based)
        m2.addConstr(reloc_expr2 <= k + 1e-6, name=f"reloc_le_{k}")
        m2.addConstr(wm_var <= eps + 1e-6, name=f"wm_cap_{eps:.3f}")

        # Rebuild distance expression on this copy
        dist2 = quicksum(
            float(distances.iloc[i_brick, j_brick]) * m2.getVarByName(f"x[{i_brick},{j_brick}]")
            for i_brick in range(m) for j_brick in range(m)
        )
        
        # Objective : minimize distance (with tiny tie-break on wm)
        m2.setObjective(dist2 + 1e-9 * wm_var, GRB.MINIMIZE)
        m2.optimize()
        
        if m2.Status == GRB.OPTIMAL:
            dist_val = dist2.getValue()
            wm_val = wm_var.X
            reloc_val = compute_relocated_expression(m2, initial_centers_zero_based).getValue()
            reloc_val = int(round(reloc_val))
            three_obj_points.append((dist_val, wm_val, reloc_val))
            print(f"dist={dist_val:.2f}, wm={wm_val:.3f}, reloc={reloc_val}")
        elif m2.Status == GRB.TIME_LIMIT:
            print("TIME LIMIT")
        else:
            print(f"No solution (Status: {m2.Status})")

    
    print(f"\n  Completed k={k}: {len([p for p in three_obj_points if p[2]==k])} solutions found for this budget")


print("\n" + "="*70)
print(f"COLLECTION COMPLETE: {len(three_obj_points)} raw solutions")
print("="*70 + "\n")

# Remove exact duplicates
three_obj_points = list({(round(d, 4), round(w, 4), r) for d, w, r in three_obj_points})
print(f"After removing duplicates: {len(three_obj_points)} unique solutions\n")

# Keep only non-dominated points in (distance, wm, reloc)
print("Computing non-dominated (Pareto optimal) solutions...")
nd_points = []
for d, w, r in three_obj_points:
    dominated = False
    for d2, w2, r2 in three_obj_points:
        if (d2, w2, r2) == (d, w, r):
            continue
        # Check if (d2,w2,r2) dominates (d,w,r)
        if d2 <= d + 1e-9 and w2 <= w + 1e-9 and r2 <= r and \
           (d2 < d - 1e-9 or w2 < w - 1e-9 or r2 < r):
            dominated = True
            break
    if not dominated:
        nd_points.append((d, w, r))

print(f"Non-dominated solutions (3 objectives): {len(nd_points)}\n")

# Pretty-print all ND solutions
print("="*70)
print("NON-DOMINATED SOLUTIONS (sorted by relocations, then wm, then distance)")
print("="*70)
for d, w, r in sorted(nd_points, key=lambda t: (t[2], t[1], t[0])):
    print(f"  Relocated={r}  |  wm={w:.3f}  |  distance={d:.2f}")
print()

# 2D visualization: color by number of relocated offices
if nd_points:
    ds = [p[0] for p in nd_points]
    ws = [p[1] for p in nd_points]
    rs = [p[2] for p in nd_points]

    plt.figure(figsize=(7, 5))
    scatter = plt.scatter(ws, ds, c=rs, cmap="viridis", s=100, edgecolors='black', linewidths=1)
    plt.xlabel("Max workload per SR (wm)")
    plt.ylabel("Total distance")
    plt.title("Step 3 – Non-dominated solutions (color = relocated offices)")
    cbar = plt.colorbar(scatter)
    cbar.set_label("Number of relocated offices")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No non-dominated points found.")


THREE-OBJECTIVE ANALYSIS: Distance vs Workload vs Relocations


######################################################################
# RELOCATION BUDGET: k = 0 (max 0 offices can be relocated)
######################################################################

[Step 1/2] Finding minimum wm with relocation budget k=0...
  Minimum wm for k=0: 1.000

[Step 2/2] Exploring 300 wm values between 1.000 and 4.000...

  [1/300] k=0, wm<=1.000... No solution (Status: 1)
  [2/300] k=0, wm<=1.010... No solution (Status: 1)
  [3/300] k=0, wm<=1.020... No solution (Status: 1)
  [4/300] k=0, wm<=1.030... No solution (Status: 1)
  [5/300] k=0, wm<=1.040... No solution (Status: 1)
  [6/300] k=0, wm<=1.050... No solution (Status: 1)
  [7/300] k=0, wm<=1.060... No solution (Status: 1)
  [8/300] k=0, wm<=1.070... No solution (Status: 1)
  [9/300] k=0, wm<=1.080... No solution (Status: 1)
  [10/300] k=0, wm<=1.090... No solution (Status: 1)
  [11/300] k=0, wm<=1.100... No solution (Status: 1)
  [12/